# Skill Dimension Loader

**Source**: `workspace.metadata.taxonomy_skill_catalog` (governed taxonomy)  
**Target**: `workspace.warehouse.dim_skill`  
**Mode**: Incremental merge (SCD Type 1)

Maintains canonical skill dimension with:
* Stable surrogate keys (skill_sk)
* Sector linkage via sector_sk foreign key
* Skill categories, aliases, and taxonomy timestamps
* Auto-refresh on taxonomy updates

In [0]:
dbutils.widgets.dropdown("force_full_refresh", "false", ["true", "false"], "Force Full Refresh")
dbutils.widgets.text("metadata_path", "/Workspace/Users/aaryan.shrivastav1403@gmail.com/LMIP/metadata", "Metadata Path")

FORCE_FULL_REFRESH = dbutils.widgets.get("force_full_refresh") == "true"
METADATA_PATH = dbutils.widgets.get("metadata_path")

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, BooleanType, LongType, DoubleType
import json

CATALOG = "workspace"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
METADATA_SCHEMA = f"{CATALOG}.metadata"

TARGET_TABLE = f"{WAREHOUSE_SCHEMA}.dim_skill"
METADATA_TABLE = f"{METADATA_SCHEMA}.dim_skill_refresh_log"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = datetime.now()

print(f"Run ID: {run_id}")
print(f"Metadata path: {METADATA_PATH}")
print(f"Force full refresh: {FORCE_FULL_REFRESH}")

In [0]:
%sql
-- Create skill dimension table if not exists
CREATE TABLE IF NOT EXISTS workspace.warehouse.dim_skill (
  skill_sk BIGINT NOT NULL COMMENT 'Surrogate key for skill',
  canonical_skill_id STRING NOT NULL COMMENT 'Natural key from taxonomy (skill_key)',
  skill_name STRING NOT NULL COMMENT 'Canonical skill name',
  skill_category STRING NOT NULL COMMENT 'Skill category (e.g., Programming, Data Analysis)',
  sector_sk BIGINT COMMENT 'Foreign key to dim_sector',
  sector_key STRING COMMENT 'Sector natural key for reference',
  skill_description STRING COMMENT 'Skill description',
  aliases STRING COMMENT 'Pipe-delimited skill aliases for matching',
  is_active BOOLEAN NOT NULL COMMENT 'Is skill active in taxonomy',
  created_at TIMESTAMP NOT NULL COMMENT 'Record creation timestamp',
  updated_at TIMESTAMP NOT NULL COMMENT 'Record last update timestamp',
  taxonomy_updated_at TIMESTAMP COMMENT 'Last taxonomy update timestamp from source',
  CONSTRAINT pk_dim_skill PRIMARY KEY (skill_sk)
)
USING DELTA
COMMENT 'Canonical skill dimension from governed taxonomy';

-- Create metadata tracking table
CREATE TABLE IF NOT EXISTS workspace.metadata.dim_skill_refresh_log (
  run_id STRING,
  skills_extracted INT,
  skills_inserted INT,
  skills_updated INT,
  force_full_refresh BOOLEAN,
  processed_at TIMESTAMP,
  status STRING,
  error_message STRING
)
USING DELTA
COMMENT 'Tracks skill dimension refresh history';

In [0]:
print("Loading skill taxonomy from taxonomy_skill_catalog table...", end=" ")

# Load skill taxonomy from table (not CSV)
skills_taxonomy_df = spark.table(f"{METADATA_SCHEMA}.taxonomy_skill_catalog")

# Load sector taxonomy to get sector context
sectors_taxonomy_df = spark.table(f"{METADATA_SCHEMA}.taxonomy_sectors")

# Load dim_sector to get sector_sk mappings
dim_sector_df = spark.table(f"{WAREHOUSE_SCHEMA}.dim_sector").select(
    F.col("sector_sk"),
    F.col("canonical_sector_id")
)

# Join skills with sectors to enrich
skill_extract_df = skills_taxonomy_df.alias("s") \
    .join(
        sectors_taxonomy_df.alias("sec"),
        F.col("s.sector_key") == F.col("sec.sector_key"),
        "left"
    ) \
    .join(
        dim_sector_df.alias("ds"),
        F.col("sec.sector_key") == F.col("ds.canonical_sector_id"),
        "left"
    ) \
    .select(
        F.col("s.skill_key").alias("canonical_skill_id"),
        F.col("s.canonical_skill").alias("skill_name"),
        F.col("s.skill_category"),
        F.col("ds.sector_sk"),
        F.col("s.sector_key"),
        F.lit(None).cast("string").alias("skill_description"),  # Placeholder for future enrichment
        F.col("s.aliases"),
        F.coalesce(F.col("s.is_active"), F.lit(True)).alias("is_active"),
        F.col("s.created_at").alias("taxonomy_created_at"),
        F.col("s.updated_at").alias("taxonomy_updated_at")
    )

skills_count = skill_extract_df.count()
print(f"✓ Loaded {skills_count} skills from taxonomy_skill_catalog")

# Get current max surrogate key
max_sk_result = spark.sql(f"SELECT COALESCE(MAX(skill_sk), 0) as max_sk FROM {TARGET_TABLE}").collect()
max_sk = max_sk_result[0]['max_sk']

print(f"Current max surrogate key: {max_sk}")

In [0]:
# Define metadata schema
metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("skills_extracted", IntegerType(), True),
    StructField("skills_inserted", IntegerType(), True),
    StructField("skills_updated", IntegerType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

try:
    print(f"Processing skills into {TARGET_TABLE}...", end=" ")
    
    # Check if table schema matches expected schema FIRST
    existing_cols = [row.col_name for row in spark.sql(f"DESCRIBE {TARGET_TABLE}").collect()]
    expected_cols = ['skill_sk', 'canonical_skill_id', 'skill_name', 'skill_category', 'sector_sk', 'sector_key', 'skill_description', 'aliases', 'is_active', 'created_at', 'updated_at', 'taxonomy_updated_at']
    schema_matches = set(existing_cols) == set(expected_cols)
    
    # Only query existing skills if schema matches
    if schema_matches:
        existing_skills = spark.sql(f"SELECT canonical_skill_id, skill_sk, created_at FROM {TARGET_TABLE}")
        
        # Join to assign keys (existing or new)
        from pyspark.sql.window import Window
        
        skills_with_keys = skill_extract_df.alias("s").join(
            existing_skills.alias("e"),
            F.col("s.canonical_skill_id") == F.col("e.canonical_skill_id"),
            "left"
        )
        
        # Assign surrogate keys
        window_spec = Window.orderBy("s.canonical_skill_id")
        
        skills_final = skills_with_keys.withColumn(
            "skill_sk",
            F.coalesce(F.col("e.skill_sk"), F.lit(max_sk) + F.row_number().over(window_spec))
        ).withColumn(
            "created_at",
            F.coalesce(F.col("e.created_at"), F.lit(run_timestamp))
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            F.col("skill_sk").cast(LongType()),
            F.col("s.canonical_skill_id"),
            F.col("s.skill_name"),
            F.col("s.skill_category"),
            F.col("s.sector_sk"),
            F.col("s.sector_key"),
            F.col("s.skill_description"),
            F.col("s.aliases"),
            F.col("s.is_active"),
            "created_at",
            "updated_at",
            F.col("s.taxonomy_updated_at")
        )
    else:
        # Schema mismatch: assign keys without joining to existing table
        from pyspark.sql.window import Window
        window_spec = Window.orderBy("canonical_skill_id")
        
        skills_final = skill_extract_df.withColumn(
            "skill_sk",
            (F.lit(max_sk) + F.row_number().over(window_spec)).cast(LongType())
        ).withColumn(
            "created_at",
            F.lit(run_timestamp)
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            "skill_sk",
            "canonical_skill_id",
            "skill_name",
            "skill_category",
            "sector_sk",
            "sector_key",
            "skill_description",
            "aliases",
            "is_active",
            "created_at",
            "updated_at",
            "taxonomy_updated_at"
        )
    
    # Create temp view for merge
    skills_final.createOrReplaceTempView("skills_to_merge")
    
    if FORCE_FULL_REFRESH or not schema_matches:
        # Full refresh: drop and recreate with new schema
        if not schema_matches:
            print(f"Schema mismatch detected. Performing full refresh...")
            spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
            spark.sql(f"""
            CREATE TABLE {TARGET_TABLE} (
              skill_sk BIGINT NOT NULL COMMENT 'Surrogate key for skill',
              canonical_skill_id STRING NOT NULL COMMENT 'Natural key from taxonomy (skill_key)',
              skill_name STRING NOT NULL COMMENT 'Canonical skill name',
              skill_category STRING NOT NULL COMMENT 'Skill category (e.g., Programming, Data Analysis)',
              sector_sk BIGINT COMMENT 'Foreign key to dim_sector',
              sector_key STRING COMMENT 'Sector natural key for reference',
              skill_description STRING COMMENT 'Skill description',
              aliases STRING COMMENT 'Pipe-delimited skill aliases for matching',
              is_active BOOLEAN NOT NULL COMMENT 'Is skill active in taxonomy',
              created_at TIMESTAMP NOT NULL COMMENT 'Record creation timestamp',
              updated_at TIMESTAMP NOT NULL COMMENT 'Record last update timestamp',
              taxonomy_updated_at TIMESTAMP COMMENT 'Last taxonomy update timestamp from source',
              CONSTRAINT pk_dim_skill PRIMARY KEY (skill_sk)
            )
            USING DELTA
            COMMENT 'Canonical skill dimension from governed taxonomy'
            """)
        else:
            spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        
        skills_final.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
        skills_inserted = skills_final.count()
        skills_updated = 0
        print(f"✓ Full refresh: {skills_inserted} skills inserted")
    else:
        # Incremental: merge
        merge_sql = f"""
        MERGE INTO {TARGET_TABLE} target
        USING skills_to_merge source
        ON target.canonical_skill_id = source.canonical_skill_id
        WHEN MATCHED THEN UPDATE SET
            target.skill_name = source.skill_name,
            target.skill_category = source.skill_category,
            target.sector_sk = source.sector_sk,
            target.sector_key = source.sector_key,
            target.skill_description = source.skill_description,
            target.aliases = source.aliases,
            target.is_active = source.is_active,
            target.updated_at = source.updated_at,
            target.taxonomy_updated_at = source.taxonomy_updated_at
        WHEN NOT MATCHED THEN INSERT *
        """
        
        spark.sql(merge_sql)
        
        # Count metrics
        skills_inserted = spark.sql(f"""
            SELECT COUNT(*) as cnt FROM skills_to_merge
            WHERE canonical_skill_id NOT IN (SELECT canonical_skill_id FROM {TARGET_TABLE})
        """).collect()[0]['cnt']
        
        skills_updated = skills_count - skills_inserted
        
        print(f"✓ Merge complete: {skills_inserted} new, {skills_updated} updated")
    
    # Log to metadata
    metadata_data = [(
        run_id,
        skills_count,
        skills_inserted,
        skills_updated,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'success',
        None
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    result = {
        "status": "success",
        "run_id": run_id,
        "skills_extracted": skills_count,
        "skills_inserted": skills_inserted,
        "skills_updated": skills_updated,
        "target_table": TARGET_TABLE,
        "metadata_table": METADATA_TABLE
    }
    
    print(json.dumps(result, indent=2))
    
except Exception as e:
    error_msg = str(e)
    print(f"✗ Error: {error_msg}")
    
    # Log failure to metadata
    metadata_data = [(
        run_id,
        skills_count if 'skills_count' in locals() else 0,
        0,
        0,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'failed',
        error_msg
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    raise

In [0]:
%sql
-- Validate skill dimension
SELECT 
  COUNT(*) as total_skills,
  COUNT(DISTINCT skill_category) as skill_categories,
  COUNT(DISTINCT sector_key) as sectors,
  COUNT(DISTINCT sector_sk) as linked_sectors,
  SUM(CASE WHEN sector_sk IS NOT NULL THEN 1 ELSE 0 END) as skills_with_sector_link,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_skills,
  MIN(created_at) as earliest_created,
  MAX(updated_at) as last_updated
FROM workspace.warehouse.dim_skill;

-- Sample skills by category with sector linkage
SELECT 
  s.skill_sk,
  s.canonical_skill_id,
  s.skill_name,
  s.skill_category,
  s.sector_key,
  sec.sector_name,
  LENGTH(s.aliases) as alias_length,
  s.is_active,
  s.created_at,
  s.updated_at,
  s.taxonomy_updated_at
FROM workspace.warehouse.dim_skill s
LEFT JOIN workspace.warehouse.dim_sector sec ON s.sector_sk = sec.sector_sk
ORDER BY s.skill_category, s.skill_name
LIMIT 25;

-- Show refresh history
SELECT 
  run_id,
  skills_extracted,
  skills_inserted,
  skills_updated,
  force_full_refresh,
  processed_at,
  status
FROM workspace.metadata.dim_skill_refresh_log
ORDER BY processed_at DESC
LIMIT 10;